In [1]:
import gym
import numpy as np
from PIL import Image
import os

# --- 設定 ---
ENV_ID = "CartPole-v1" # 環境
GIF_FILENAME = "cartpole_random_episode.gif"
FRAME_DURATION_MS = 40
MAX_FRAMES = 500 # 最多儲存的幀數 (CartPole 通常一個 episode 不會太長)

# --- 創建環境 ---
# render_mode='rgb_array' 模式是儲存 GIF 必需的
try:
    env = gym.make(ENV_ID, render_mode='rgb_array')
    print(f"成功創建環境 {ENV_ID}")
    print(f"動作空間：{env.action_space}") # CartPole 的動作空間是離散的 (左/右)
    print(f"狀態空間：{env.observation_space}") # CartPole 的狀態空間是連續的
except Exception as e:
    print(f"創建環境時發生錯誤: {e}")
    exit()


# --- 執行一個 Episode 並收集幀 ---
frames = []
print("開始執行一個隨機動作的 episode...")

observation, info = env.reset(seed=42) # 可以設定 seed 讓結果重現
done = False
frame_count = 0
total_reward = 0

while not done and frame_count < MAX_FRAMES:
    # 渲染當前幀並添加到列表中
    frame = env.render()
    frames.append(frame)

    # 選擇一個隨機動作 (CartPole 的動作空間是離散的，通常是 0 或 1)
    action = env.action_space.sample()

    # 執行動作
    observation, reward, terminated, truncated, info = env.step(action)

    total_reward += reward # 累積獎勵

    # 檢查 episode 是否結束
    done = terminated or truncated

    frame_count += 1

print(f"Episode 結束。總共收集了 {frame_count} 幀。累計獎勵：{total_reward}")

# --- 關閉環境 ---
env.close()

# --- 將幀儲存為 GIF ---
if frames:
    print(f"正在將 {len(frames)} 幀儲存為 GIF: {GIF_FILENAME}...")
    # 將 NumPy 陣列轉換為 PIL Image 物件
    image_objects = [Image.fromarray(frame) for frame in frames]

    # 儲存 GIF
    if image_objects:
        # 如果只有一幀，直接保存為單幀 GIF
        if len(image_objects) == 1:
             image_objects[0].save(GIF_FILENAME, save_all=True)
        else:
            image_objects[0].save(
                GIF_FILENAME,
                save_all=True,
                append_images=image_objects[1:], # 從第二幀開始添加到第一幀後面
                optimize=False,
                duration=FRAME_DURATION_MS,
                loop=0 # 0 表示無限循環
            )
        print(f"GIF 已成功儲存到 {GIF_FILENAME}")
    else:
        print("沒有收集到任何幀，無法儲存 GIF。")
else:
    print("沒有收集到任何幀，無法儲存 GIF。")

成功創建環境 CartPole-v1
動作空間：Discrete(2)
狀態空間：Box([-4.8000002e+00 -3.4028235e+38 -4.1887903e-01 -3.4028235e+38], [4.8000002e+00 3.4028235e+38 4.1887903e-01 3.4028235e+38], (4,), float32)
開始執行一個隨機動作的 episode...


c:\Users\jhbai\Anaconda\envs\MachineLearningEnv\Lib\site-packages\gym\utils\passive_env_checker.py:233: DeprecationWarning: `np.bool8` is a deprecated alias for `np.bool_`.  (Deprecated NumPy 1.24)
  if not isinstance(terminated, (bool, np.bool8)):


Episode 結束。總共收集了 24 幀。累計獎勵：24.0
正在將 24 幀儲存為 GIF: cartpole_random_episode.gif...
GIF 已成功儲存到 cartpole_random_episode.gif


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import random
import numpy as np
from collections import deque


class DQN(nn.Module):
    """
    DQN 神經網絡模型，輸入為環境狀態，輸出為各行動的 Q 值
    """
    def __init__(self, state_dim, action_dim, hidden_dim=128):
        super(DQN, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, action_dim)
        )

    def forward(self, x):
        return self.net(x)


class DQNAgent:
    """
    DQN Agent，包含訓練與推論方法
    方法:
      - take_action: 根據 epsilon-greedy 選擇動作
      - fit: 在環境上執行訓練
      - save / load: 模型參數存取
    """
    def __init__(
        self,
        state_dim,
        action_dim,
        hidden_dim=128,
        lr=1e-3,
        gamma=0.99,
        epsilon=1.0,
        epsilon_min=0.01,
        epsilon_decay=0.995,
        buffer_size=10000,
        batch_size=64,
        target_update_freq=1000,
        device=None
    ):
        # 強化學習超參數
        self.gamma = gamma
        self.epsilon = epsilon
        self.epsilon_min = epsilon_min
        self.epsilon_decay = epsilon_decay
        self.batch_size = batch_size
        self.target_update_freq = target_update_freq

        # 裝置 (CPU / GPU)
        self.device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")

        # 線上網絡與目標網絡
        self.online_net = DQN(state_dim, action_dim, hidden_dim).to(self.device)
        self.target_net = DQN(state_dim, action_dim, hidden_dim).to(self.device)
        self.target_net.load_state_dict(self.online_net.state_dict())
        self.target_net.eval()

        # 優化器與損失函數
        self.optimizer = optim.Adam(self.online_net.parameters(), lr=lr)
        self.loss_fn = nn.MSELoss()
        self.best_reward = -np.inf
        self.model = None

        # 經驗回放緩衝區
        self.memory = deque(maxlen=buffer_size)
        self.learn_step_counter = 0

    def take_action(self, state, greedy=False):
        """
        選擇動作:
        - 訓練時使用 epsilon-greedy
        - 推論時 greedy=True，僅依據最大 Q 值
        """
        state = torch.FloatTensor(state).unsqueeze(0).to(self.device)
        if not greedy and random.random() < self.epsilon:
            return random.randrange(self.online_net.net[-1].out_features)
        with torch.no_grad():
            q_values = self.online_net(state)
        return q_values.argmax().item()

    def store_transition(self, state, action, reward, next_state, done):
        """存儲一筆經驗到回放緩衝區"""
        self.memory.append((state, action, reward, next_state, done))

    def sample_memory(self):
        """隨機抽樣一個 batch 的經驗"""
        batch = random.sample(self.memory, self.batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return np.vstack(states), actions, rewards, np.vstack(next_states), dones

    def update(self):
        """根據 DQN 算法更新網絡參數"""
        if len(self.memory) < self.batch_size:
            return

        states, actions, rewards, next_states, dones = self.sample_memory()

        # 轉為張量
        states = torch.FloatTensor(states).to(self.device)
        actions = torch.LongTensor(actions).unsqueeze(1).to(self.device)
        rewards = torch.FloatTensor(rewards).unsqueeze(1).to(self.device)
        next_states = torch.FloatTensor(next_states).to(self.device)
        dones = torch.FloatTensor(dones).unsqueeze(1).to(self.device)

        # 計算 Q(s,a)
        q_eval = self.online_net(states).gather(1, actions)
        # 計算目標 Q value
        with torch.no_grad():
            q_next = self.target_net(next_states).max(1)[0].unsqueeze(1)
            q_target = rewards + self.gamma * q_next * (1 - dones)

        loss = self.loss_fn(q_eval, q_target)

        # 反向傳播
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        # 定期更新目標網絡
        self.learn_step_counter += 1
        if self.learn_step_counter % self.target_update_freq == 0:
            self.target_net.load_state_dict(self.online_net.state_dict())

    def fit(self, env, num_episodes=500):
        """
        在給定環境上訓練 DQN Agent
        :param env: 包含 reset, step, render 等方法的環境物件
        :param num_episodes: 訓練迭代次數
        """
        self.online_net.train()
        from IPython.display import clear_output, display
        for episode in range(1, num_episodes + 1):
            state = env.reset()
            done = False
            total_reward = 0
            while not done:
                action = self.take_action(state)
                next_state, reward, done, _ = env.step(action)
                self.store_transition(state, action, reward, next_state, done)
                self.update()
                state = next_state
                total_reward += reward
            # epsilon 衰減
            self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)
            print(f"Episode {episode}, Total Reward: {total_reward}, Epsilon: {self.epsilon:.3f}")
            if total_reward >= self.best_reward:
                self.best_reward = total_reward
                self.save("best_dqn.pth")
                self.model = self.online_net.state_dict()
                print(f"新最佳獎勵: {self.best_reward}，已儲存模型")
                
            clear_output(wait=True)
        self.online_net.load_state_dict(self.model)
        self.online_net.eval()

    def save(self, path):
        """儲存線上網絡參數"""
        torch.save(self.online_net.state_dict(), path)

    def load(self, path):
        """載入模型參數並同步到目標網絡"""
        self.online_net.load_state_dict(torch.load(path, map_location=self.device))
        self.target_net.load_state_dict(self.online_net.state_dict())


In [3]:
class Env:
    def __init__(self):
        self.env = gym.make("CartPole-v1", render_mode='rgb_array')
        self.state = self.env.reset()[0]
        self.done = False
    def step(self, action):
        self.state, reward, terminated, truncated, info = self.env.step(action)
        self.done = terminated or truncated
        return self.state, reward, self.done, info
    def reset(self):
        self.state = self.env.reset()[0]
        self.done = False
        return self.state
    def render(self):
        return self.env.render()
    def close(self):
        self.env.close()

In [6]:
agent = DQNAgent(
    state_dim=env.observation_space.shape[0],
    action_dim=env.action_space.n,
    hidden_dim=128,
    lr=1e-3,
    gamma=0.99,
    epsilon=0.1,
    epsilon_min=0.01,
    epsilon_decay=0.995,
    buffer_size=10000,
    batch_size=64,
    target_update_freq=100,
    device=None
)
agent.fit(Env(), num_episodes=600)

Episode 600, Total Reward: 500.0, Epsilon: 0.010
新最佳獎勵: 500.0，已儲存模型


In [7]:
frames = []
ENV = Env()
done = False
state = ENV.reset()
while not done:
    action = agent.take_action(state, greedy=False)
    next_state, reward, done, info = ENV.step(action)
    state = next_state
    frame = ENV.render()
    frames.append(frame)
ENV.close()

image_objects = [Image.fromarray(np.uint8(frame)) for frame in frames]
image_objects[0].save(
                    GIF_FILENAME,
                    save_all=True,
                    append_images=image_objects[1:],
                    optimize=False, # 可以嘗試 optimize=True，但有時會導致顏色或透明度問題
                    duration=FRAME_DURATION_MS,
                    loop=0
                )